# Public Restroom Map: open data + crowdsourced ratings

**Team project** for *Introduction to Python for Business Analytics* (NCCU, Spring 2024).

Taiwan's Ministry of Environment publishes where every public restroom is, but not what condition it is in.
This notebook combines the official list with ratings from a short visitor survey and puts both on one interactive map.

Pipeline:

1. Download the national list of public restrooms from the open-data API.
2. Read survey responses from a Google Sheet.
3. Turn each response into a 0-100 condition score and aggregate by restroom.
4. Join the scores to the official list and draw the map.

## Setup

In [ ]:
import os
from io import StringIO

import folium
import gspread
import pandas as pd
import requests
from folium.plugins import MarkerCluster

# Secrets and private links come from environment variables and are never stored in the notebook.
EPA_API_KEY = os.environ["EPA_API_KEY"]
SERVICE_ACCOUNT_FILE = os.environ["GOOGLE_SERVICE_ACCOUNT_FILE"]
SURVEY_SHEET_URL = os.environ["SURVEY_SHEET_URL"]

API_URL = "https://data.moenv.gov.tw/api/v2/fac_p_07"  # public restroom register
PAGE_SIZE = 1000

DISTRICT = "文山區"                   # Wenshan District, Taipei
VILLAGES = ["政大里", "景美里"]        # neighbourhoods covered by the survey
CAMPUS_LOCATION = [24.9875, 121.5764]

## 1. Download the official restroom list

The API returns at most 1,000 rows per request, so we page through it until a short page signals the end.

In [ ]:
def fetch_restrooms(api_key: str, page_size: int = PAGE_SIZE) -> pd.DataFrame:
    """Download every row of the public restroom register."""
    pages = []
    offset = 0
    while True:
        response = requests.get(
            API_URL,
            params={
                "api_key": api_key,
                "format": "csv",
                "limit": page_size,
                "offset": offset,
                "sort": "ImportDate desc",
            },
            timeout=30,
        )
        response.raise_for_status()
        page = pd.read_csv(StringIO(response.text))
        pages.append(page)
        if len(page) < page_size:
            break
        offset += page_size
    return pd.concat(pages, ignore_index=True)

In [ ]:
restrooms = fetch_restrooms(EPA_API_KEY)
print(f"{len(restrooms):,} public restrooms nationwide")

local_restrooms = restrooms[restrooms["village"].isin(VILLAGES)].copy()
print(f"{len(local_restrooms)} in the surveyed neighbourhoods")
local_restrooms.head()

## 2. Read the survey responses

Visitors fill in a Google Form. Each response names a restroom, answers five yes/no questions about its condition and can add a short comment.

In [ ]:
def load_survey(sheet_url: str, service_account_file: str, worksheet: str = "Survey Sheet") -> pd.DataFrame:
    """Read all form responses from the linked Google Sheet."""
    client = gspread.service_account(service_account_file)
    rows = client.open_by_url(sheet_url).worksheet(worksheet).get_all_values()
    header = [name.strip() for name in rows[0]]
    return pd.DataFrame(rows[1:], columns=header)

In [ ]:
survey_raw = load_survey(SURVEY_SHEET_URL, SERVICE_ACCOUNT_FILE)
print(f"{len(survey_raw)} survey responses")

## 3. Score each response

Four questions describe a problem ("is the flush leaking?"), so *no* is the good answer. One asks whether there is enough toilet paper, so *yes* is the good answer.
A response's score is the share of good answers, from 0 to 100.

In [ ]:
YES, NO = "是", "否"

# Survey question -> the answer that means the restroom is in good condition.
GOOD_ANSWER = {
    "大小便器嚴重破損或缺水": NO,     # fixtures badly damaged or without water
    "沖水設備有漏水或阻塞": NO,       # flush leaking or blocked
    "洗手設備有漏水或水壓過小": NO,   # tap leaking or weak water pressure
    "廁所配有足夠的衛生紙": YES,      # enough toilet paper
    "垃圾桶滿溢": NO,                 # bins overflowing
}
COMMENT_COLUMN = "簡短評論"           # short free-text comment
AREA_COLUMNS = ["請選擇縣市", "請選擇行政區", "請選擇里"]  # county, district, village
TIMESTAMP_COLUMN = "時間戳記"


def score_survey(survey_raw: pd.DataFrame) -> pd.DataFrame:
    """Return one row per response: restroom name, 0-100 score and comment."""
    known = set(GOOD_ANSWER) | set(AREA_COLUMNS) | {COMMENT_COLUMN, TIMESTAMP_COLUMN}

    # The form shows a different restroom list for each village, so the name
    # can sit in any of several columns. Take the first one that is filled in.
    is_name_column = [column not in known for column in survey_raw.columns]
    names = (
        survey_raw.loc[:, is_name_column]
        .replace("", pd.NA)
        .bfill(axis=1)
        .iloc[:, 0]
    )

    good_answers = pd.DataFrame({
        question: survey_raw[question].str.strip() == good
        for question, good in GOOD_ANSWER.items()
    })

    scored = pd.DataFrame({
        "name": names,
        "score": good_answers.mean(axis=1) * 100,
        "comment": survey_raw[COMMENT_COLUMN].str.strip(),
    })
    return scored.dropna(subset=["name"])


def summarise_by_restroom(scored: pd.DataFrame) -> pd.DataFrame:
    """Average score, number of responses and all comments for each restroom."""
    return (
        scored.groupby("name")
        .agg(
            avg_score=("score", "mean"),
            responses=("score", "size"),
            comments=("comment", lambda c: [text for text in c if text]),
        )
        .reset_index()
    )

In [ ]:
ratings = summarise_by_restroom(score_survey(survey_raw))
ratings.sort_values("avg_score").head(10)

## 4. Join and map

A left join keeps every official restroom, rated or not. Unrated ones are drawn in grey so it is obvious where more survey responses are needed.

In [ ]:
def score_colour(score) -> str:
    if pd.isna(score):
        return "gray"
    if score >= 80:
        return "green"
    if score >= 50:
        return "orange"
    return "red"


def popup_html(row: pd.Series) -> str:
    if pd.isna(row["avg_score"]):
        rating = "No survey responses yet"
        comments = ""
    else:
        rating = f"{row['avg_score']:.0f} / 100 ({int(row['responses'])} responses)"
        comments = "".join(f"<li>{text}</li>" for text in row["comments"])
    return f"""
    <div style="width: 220px">
      <h4 style="margin: 0 0 6px">{row['name']}</h4>
      <p style="margin: 0"><b>Address:</b> {row['address']}</p>
      <p style="margin: 0"><b>Type:</b> {row['type']}</p>
      <p style="margin: 0"><b>Condition:</b> {rating}</p>
      <ul style="margin: 4px 0 0; padding-left: 18px">{comments}</ul>
    </div>
    """


def build_map(restrooms_with_ratings: pd.DataFrame, centre=CAMPUS_LOCATION, zoom: int = 15) -> folium.Map:
    """One marker per restroom, coloured by its average survey score."""
    restroom_map = folium.Map(location=centre, zoom_start=zoom, tiles="openstreetmap")
    cluster = MarkerCluster().add_to(restroom_map)
    for _, row in restrooms_with_ratings.iterrows():
        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            popup=folium.Popup(popup_html(row), max_width=300),
            icon=folium.Icon(color=score_colour(row["avg_score"]), icon="info-sign"),
        ).add_to(cluster)
    return restroom_map

In [ ]:
restrooms_with_ratings = local_restrooms.merge(ratings, on="name", how="left")
rated = restrooms_with_ratings["avg_score"].notna().sum()
print(f"{rated} of {len(restrooms_with_ratings)} local restrooms have at least one rating")

restroom_map = build_map(restrooms_with_ratings)
restroom_map.save("restroom_map.html")
restroom_map

## What we would improve

- Match survey answers to restrooms by an ID or coordinates instead of by name.
- Show how each restroom's score changes over time, not only the average.
- Extend the survey beyond the two neighbourhoods around campus.